In [13]:
# Examine HDF5 files written by `record_info_at_angle` in `batch_process.py`

'''
angle_{angle}.hdf5
  dc_{dc}/
    output_1_bright
    output_2_bright
    output_3_dark
    output_4_dark
    chopped
'''

# Each leaf is an Astropy `QTable` dataset. Companion `.__table_column_meta__` datasets store units and column metadata when `serialize_meta=True`.

'\nangle_{angle}.hdf5\n  dc_{dc}/\n    output_1_bright\n    output_2_bright\n    output_3_dark\n    output_4_dark\n    chopped\n'

In [14]:
from pathlib import Path
import h5py
from astropy.table import QTable

In [15]:
file_name_abs = (
    '/Users/eckhartspalding/Documents/git.repos/life_detectors/param_sweeps/20260408_M_stars/'
    'temp_s2n_sweep_planet_index_0000000_Nuniverse_10_Nstar_2425_dist_27.93_Rp_0.9781_Rs_0.421_Ts_3470_L_0.05_z_5.9973_eclip_lon_279.332_eclip_lat_36.0901_Stype_M/angle_0.hdf5'
)

In [19]:
file_name_abs = '/Users/eckhartspalding/Documents/git.repos/life_detectors/hdf5_testing/temp_s2n_sweep_planet_index_0000000_Nuniverse_10_Nstar_2425_dist_27.93_Rp_0.9781_Rs_0.421_Ts_3470_L_0.05_z_5.9973_eclip_lon_279.332_eclip_lat_36.0901_Stype_M/angle_0.hdf5'

In [20]:
def print_hdf5_structure(file_path, *, show_meta: bool = True, show_columns: bool = True):
    """
    Print groups, QTable datasets, column names, and table metadata for an angle HDF5 file.

    Table metadata (e.g. angle_deg, dark_current_e_pix_s) is restored via QTable.read and
  comes from the .__table_column_meta__ sidecar written with serialize_meta=True.
    """
    file_path = Path(file_path)
    print(f"File: {file_path}")
    print(f"Size: {file_path.stat().st_size / 1024:.1f} KiB\n")

    with h5py.File(file_path, "r") as h5:
        def visitor(name, obj):
            # Astropy stores units + table.meta in this sidecar; shown under the parent table.
            if name.endswith(".__table_column_meta__"):
                return

            if isinstance(obj, h5py.Group):
                print(f"GROUP {name or '/'}")
                return

            if isinstance(obj, h5py.Dataset):
                fields = obj.dtype.names or ()
                print(f"TABLE {name}  rows={obj.shape[0]}  n_cols={len(fields)}")
                if show_columns and fields:
                    for field in fields:
                        print(f"        - {field}")
                if show_meta:
                    tbl = QTable.read(file_path, path=name)
                    if tbl.meta:
                        for key, val in tbl.meta.items():
                            print(f"        meta {key}: {val}")

        print("Structure:")
        h5.visititems(visitor)

In [21]:
print_hdf5_structure(file_name_abs)

File: /Users/eckhartspalding/Documents/git.repos/life_detectors/hdf5_testing/temp_s2n_sweep_planet_index_0000000_Nuniverse_10_Nstar_2425_dist_27.93_Rp_0.9781_Rs_0.421_Ts_3470_L_0.05_z_5.9973_eclip_lon_279.332_eclip_lat_36.0901_Stype_M/angle_0.hdf5
Size: 13.8 KiB

Structure:
GROUP dc_0
TABLE dc_0/output_1_bright  rows=61  n_cols=8
        - wavel_bin_num
        - wavel_bin_center
        - wavel_bin_width
        - n_pix_per_wavel_bin
        - instrum_dark_current_for_wavel_bin_and_integration_adu_tot
        - instrum_read_noise_for_wavel_bin_and_integration_adu_tot
        - astro_star_flux_adu_sec_for_wavel_bin_and_integration_tot
        - astro_exoplanet_model_10pc_flux_adu_sec_for_wavel_bin_and_integration_tot
        meta angle_deg: 0.0
        meta dark_current_e_pix_s: 0.0


In [18]:
# Note: dc groups use Python's :g format, so dc_rate=0.0 is stored as "dc_0" (not "dc_0.0").
dc_rate = 0.0
hdf5_path = f"dc_{dc_rate:g}/output_1_bright"

t = QTable.read(file_name_abs, path=hdf5_path)
print("table.meta:", t.meta)
print()
print(t)
print()
print("wavel unit:", t["wavel_bin_center"].unit)
print("star flux unit:", t["astro_star_flux_adu_sec_for_wavel_bin_and_integration_tot"].unit)

table.meta: OrderedDict([('angle_deg', 0.0), ('dark_current_e_pix_s', 0.0)])

wavel_bin_num ...
              ...
------------- ...
            0 ...
            1 ...
            2 ...
            3 ...
            4 ...
            5 ...
            6 ...
            7 ...
            8 ...
            9 ...
          ... ...
           51 ...
           52 ...
           53 ...
           54 ...
           55 ...
           56 ...
           57 ...
           58 ...
           59 ...
           60 ...
Length = 61 rows

wavel unit: um
star flux unit: adu
